# Data Clean

This notebook contains the steps for data cleaning, integration, and preparation, with explicit handling of date formats.

In [8]:

# Import required libraries
import pandas as pd
from sklearn.impute import SimpleImputer

# Function to perform data integration and cleaning
def data_integration_and_cleaning(tables: dict[str, pd.DataFrame]):
    """
    Performs data integration and cleaning:
    - Ensures consistent formats across tables.
    - Matches entities (e.g., IDs).
    - Cleans up missing values with advanced techniques.
    
    Args:
        tables (dict[str, pd.DataFrame]): Dictionary of DataFrames.
    
    Returns:
        dict[str, pd.DataFrame]: Updated DataFrames.
    """
    # Step 1: Convert formats to ensure consistency
    for name, df in tables.items():
        print(f"Processing table: {name}")
        if 'year' in df.columns:
            df['year'] = pd.to_datetime(df['year'], format='%Y', errors='coerce')
        
        if 'birthDate' in df.columns:
            df['birthDate'] = pd.to_datetime(df['birthDate'], format='%Y-%m-%d', errors='coerce')
        
        if 'deathDate' in df.columns:
            df['deathDate'] = pd.to_datetime(df['deathDate'], format='%Y-%m-%d', errors='coerce')

        # Remove problematic columns like 'divID'
        if 'divID' in df.columns:
            df = df.drop(columns=['divID'])

        tables[name] = df

    # Step 2: Match entities across tables (e.g., playerID consistency)
    players_ids = set(tables['players']['bioID'])
    tables['awards_players'] = tables['awards_players'][
        tables['awards_players']['playerID'].isin(players_ids)
    ]

    # Step 3: Fill missing values with advanced techniques (if applicable)
    imputer = SimpleImputer(strategy='median')
    for name, df in tables.items():
        for col in df.select_dtypes(include=['float64', 'int64']).columns:
            if df[col].isnull().sum() > 0:
                try:
                    df[col] = imputer.fit_transform(df[[col]])
                except ValueError:
                    print(f"Skipping column '{col}' in table '{name}' due to insufficient data.")

        tables[name] = df

    return tables

# Example of loading raw data
tables = {
    "awards_players": pd.read_csv('../data/development_data/awards_players.csv'),
    "coaches": pd.read_csv('../data/development_data/coaches.csv'),
    "players": pd.read_csv('../data/development_data/players.csv'),
    "players_teams": pd.read_csv('../data/development_data/players_teams.csv'),
    "series_post": pd.read_csv('../data/development_data/series_post.csv'),
    "teams": pd.read_csv('../data/development_data/teams.csv'),
    "teams_post": pd.read_csv('../data/development_data/teams_post.csv'),
}

# Apply the integration and cleaning function
cleaned_and_integrated_data = data_integration_and_cleaning(tables)

# Display an example of cleaned data
cleaned_and_integrated_data['players'].head()


Processing table: awards_players
Processing table: coaches
Processing table: players
Processing table: players_teams
Processing table: series_post
Processing table: teams
Processing table: teams_post


,bioID,pos,firstseason,lastseason,height,weight,college,collegeOther,birthDate,deathDate
0,abrahta01w,C,0,0,74.0,190,George Washington,NaN,1975-09-27,NaT
1,abrossv01w,F,0,0,74.0,169,Connecticut,NaN,1980-07-09,NaT
2,adairje01w,C,0,0,76.0,197,George Washington,NaN,1986-12-19,NaT
3,adamsda01w,F-C,0,0,73.0,239,Texas A&M,Jefferson College (JC),1989-02-19,NaT
4,adamsjo01w,C,0,0,75.0,180,New Mexico,NaN,1981-05-24,NaT
